# D3QN V3 — Mini16/Val4 learning-check (fresh training)

**Đây là thí nghiệm V3 độc lập** sau phân tích thất bại/không ổn định của V2.

- Khóa **Train16 / Val4** từ BDD100K, **không dùng Test30**.
- D3QN v3: 3-step return (γ=1), Polyak target, khám phá action chậm hơn, state nhớ hai ΔQP ROI/BG riêng.
- 20 episode × 256 frame = **5.120 transitions**. Train16 được phủ đủ trong 16 episode đầu.
- Val4 tại **120 kbps** sau mỗi 1.024 bước (5 checkpoint).
- So sánh QPA, global -2, fixed action19 (+1,+2), fixed action2 (-2,0).
- **Best checkpoint chọn theo mean Val4 objective**, không tự coi checkpoint cuối là tốt nhất.
- Chạy unit tests + native VVenC/QG + VVdeC preflight trước training.
- Không cam kết V3 tốt hơn V2; chỉ là một thí nghiệm kiểm chứng giả thuyết.
- Notebook chỉ dùng **một GPU** cho detector/agent, có thể chạy trên Kaggle T4×2 hoặc GPU tương thích khác.

**Kaggle:** Bật Accelerator=GPU, Internet=On, gắn BDD100K Driving Video with Object Tracking (MOV files).

In [ ]:
import json, os, sys, time, shutil, subprocess, zipfile, traceback
from pathlib import Path
from statistics import mean

# Commit PINNED: never silently run a future branch revision.
SOURCE_COMMIT="07a0eacb6793e368920ae0b89b9b51d06b5b56b2"
BRANCH='experiment/d3qn-native-mdp-v2'
REPO=Path('/kaggle/working/rl_vvc_d3qn_v3_source')
TOOLS=Path('/kaggle/working/rl_vvc_d3qn_v3_tools')
RUN=Path('/kaggle/working/rl_vvc_d3qn_v3_mini16_val4')
SAFE=Path('/kaggle/working/D3QN_V3_MINI16_VAL4_LEARNINGCHECK_SAFE.zip')
INPUT_ROOT=Path('/kaggle/input')
SEED=20261111
DEVICE='cuda:0'
SESSION_HOURS=11.75
if 'D3QN_V3_SESSION_STARTED' not in globals():
    D3QN_V3_SESSION_STARTED=time.time()
MILESTONES=[1024,2048,3072,4096,5120]
VAL_RATE=120

# Optional: mount a previous V3 SAFE ZIP as Kaggle Input, then set its path.
# Never use V2 ZIP/checkpoint: incompatible state and replay protocol.
RESUME_SAFE_ZIP=None
print('PINNED_SOURCE',SOURCE_COMMIT)
print('MINI_TRAIN',16,'MINI_VAL',4,'TEST_USED',False)
print('MILESTONES',MILESTONES,'VAL_RATE',VAL_RATE)

In [ ]:
def live(argv,cwd=None):
    command=list(map(str,argv))
    print('$',' '.join(command),flush=True)
    p=subprocess.Popen(command,cwd=str(cwd) if cwd else None,
                       stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                       text=True,bufsize=1)
    try:
        for line in p.stdout:
            print(line,end='',flush=True)
        status=p.wait()
    finally:
        if p.poll() is None:
            p.terminate()
            p.wait()
    if status:
        raise RuntimeError('Command failed with exit status '+str(status))
    return status

def loadjson(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def savejson(path,payload):
    path=Path(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(payload,indent=2,allow_nan=False),encoding='utf-8')
    tmp.replace(path)

if not (REPO/'.git').exists():
    live(['git','clone','--branch',BRANCH,'--depth','1',
          'https://github.com/youngan2810/RL-VVC.git',REPO])
live(['git','fetch','--no-tags','--depth','1','origin',SOURCE_COMMIT],REPO)
live(['git','checkout','--detach',SOURCE_COMMIT],REPO)
checked=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
assert checked==SOURCE_COMMIT,'Source checkout does not match the research pin'
assert not subprocess.check_output(
    ['git','status','--porcelain','--untracked-files=no'],cwd=REPO,text=True).strip(), 'Tracked source modified'

# Dataset hash gate; this checks only TRAIN and VALIDATION MOV, never TEST.
verify_code="""from scripts.rl_rc_dot.data import VideoCatalog
cat=VideoCatalog('/kaggle/input',mode='mini',seed=20261111)
assert len(cat.selected['train'])==16 and len(cat.selected['val'])==4
assert set(cat.selected['train']).isdisjoint(set(cat.selected['val']))
cat.check_available(('train','val'))
for split in ('train','val'):
    for video in cat.selected[split]:
        cat.source(video)  # SHA256 verify only these 20 locked videos
print('MINI16_VAL4_SHA256_PASS', len(cat.selected['train']),len(cat.selected['val']))
"""
live([sys.executable,'-c',verify_code],REPO)
print('SOURCE_AND_DATA_LOCK_PASS')

In [ ]:
import torch
assert torch.cuda.is_available(),'Kaggle Accelerator: hãy bật GPU trước khi chạy'
assert torch.cuda.device_count()>=1,'Không tìm thấy GPU CUDA'
print('CUDA',torch.__version__,torch.version.cuda,'GPUS',torch.cuda.device_count())

if not (TOOLS/'TOOLCHAIN.json').is_file():
    live([sys.executable,'-m','scripts.rl_rc_dot.setup_kaggle',
          '--repo',REPO,'--source-commit',SOURCE_COMMIT,
          '--output',TOOLS,'--jobs','2','--install-system'],REPO)
toolchain=loadjson(TOOLS/'TOOLCHAIN.json')
assert toolchain['source_commit']==SOURCE_COMMIT
for field in ('encoder','decoder','weights'):
    assert Path(toolchain[field]).is_file(), 'Missing toolchain asset '+field
print('TOOLCHAIN_PASS',{k:toolchain[k] for k in ('encoder','decoder','weights')})

In [ ]:
# Mandatory: D3QN CPU contracts, YUV->RGB parity, real VVenC QG, VVdeC decode.
live([sys.executable,'-m','unittest',
      'scripts.rl_rc_dot.test_d3qn_native_v2',
      'scripts.rl_rc_dot.test_d3qn_native_v3','-v'],REPO)
live([sys.executable,'-c',
      'from scripts.rl_rc_dot.pixels import verify_conversion; print(verify_conversion())'],REPO)
RUN.mkdir(parents=True,exist_ok=True)
preflight=RUN/'native_preflight32'
if not (preflight/'PREFLIGHT.json').is_file():
    assert not preflight.exists(),'Incomplete preflight folder; inspect failure before rerun'
    live([sys.executable,'-m','scripts.rl_rc_dot.preflight',
          '--encoder',toolchain['encoder'],'--decoder',toolchain['decoder'],
          '--output',preflight,'--frames','32','--threads','2'],REPO)
gate=loadjson(preflight/'PREFLIGHT.json')
assert gate['gate']=='REAL_VVC_STATE_ACTION_AU_DECODE_PASS'
print('ALL_MANDATORY_PREFLIGHTS_PASS')

In [ ]:
def runner_args(phase,output,checkpoint=None,train_steps=1024,methods=None):
    argv=[sys.executable,'-u','-m','scripts.rl_rc_dot.d3qn_native_runner_v3',
          '--phase',phase,'--input-root',INPUT_ROOT,
          '--encoder',toolchain['encoder'],'--decoder',toolchain['decoder'],
          '--yolo-repo',toolchain['yolo_repo'],'--weights',toolchain['weights'],
          '--output',output,'--device',DEVICE,
          '--seed',str(SEED),'--encoder-threads','2',
          '--val-videos','4','--val-start','0','--val-rates',str(VAL_RATE),
          '--train-steps',str(train_steps),
          '--n-step','3','--target-tau','0.005','--learning-rate','0.00005',
          '--epsilon-floor','0.20','--epsilon-decay-steps','10240',
          '--max-hours',str(SESSION_HOURS),
          '--session-start-utc',str(D3QN_V3_SESSION_STARTED)]
    if checkpoint is not None:
        argv+=['--checkpoint-in',str(checkpoint)]
    if methods:
        argv+=['--val-methods',methods]
    return argv

CHECKPOINTS=RUN/'checkpoints'
CHECKPOINTS.mkdir(exist_ok=True)
LATEST=CHECKPOINTS/'latest_checkpoint.pt'
BEST=CHECKPOINTS/'best_checkpoint.pt'
TRAIN_HISTORY=RUN/'train_history.json'
MONITOR_HISTORY=RUN/'monitor_history.json'
BASELINE=RUN/'baseline.json'
MANIFEST=RUN/'learning_manifest.json'

# Optional cross-Kaggle-session resume from this notebook's own ZIP.
if RESUME_SAFE_ZIP is not None:
    zpath=Path(RESUME_SAFE_ZIP)
    assert zpath.is_file(),'RESUME_SAFE_ZIP not found'
    allowed=('baseline.json','train_history.json','monitor_history.json',
             'learning_manifest.json','checkpoints/latest_checkpoint.pt',
             'checkpoints/best_checkpoint.pt')
    with zipfile.ZipFile(zpath) as bundle:
        if MANIFEST.exists() or TRAIN_HISTORY.exists():
            raise FileExistsError('Resume destination is not empty')
        for item in allowed:
            if item in bundle.namelist():
                target=RUN/item
                target.parent.mkdir(parents=True,exist_ok=True)
                target.write_bytes(bundle.read(item))
    restored=loadjson(MANIFEST)
    assert restored['source_commit']==SOURCE_COMMIT and restored['seed']==SEED
    assert restored['protocol']=='native-masked-d3qn-qp/v3'
    print('V3_RESUME_SAFE_IMPORT_PASS',restored['completed_steps'])

train_history=loadjson(TRAIN_HISTORY) if TRAIN_HISTORY.is_file() else []
monitor_history=loadjson(MONITOR_HISTORY) if MONITOR_HISTORY.is_file() else []
completed_steps=int(train_history[-1]['total_steps']) if train_history else 0
assert completed_steps%256==0 and 0<=completed_steps<=5120
if completed_steps:
    assert LATEST.is_file(),'Training history exists but latest V3 checkpoint is missing'
print('TRAINING_STATE',completed_steps,'VAL_MILESTONES',[x['steps'] for x in monitor_history])

In [ ]:
# Baselines: QPA, fixed global -2, V2 dominant action19, and ROI-protecting action2.
CONTROL_METHODS='qpa_anchor,global_minus2,roi_plus1_bg_plus2,roi_minus2_bg_zero'
if BASELINE.is_file():
    baseline=loadjson(BASELINE)
else:
    base_run=RUN/'baseline_native_val4'
    assert not base_run.exists(),'Incomplete previous baseline: inspect logs before retry'
    live(runner_args('probe',base_run,methods=CONTROL_METHODS),REPO)
    assert loadjson(base_run/'result.json')['status']=='probe_complete','Baseline not complete'
    baseline=loadjson(base_run/'validation.json')
    savejson(BASELINE,baseline)
assert len(baseline)==16
assert len(set(x['video'] for x in baseline))==4
assert set(x['method'] for x in baseline)==set(CONTROL_METHODS.split(','))
assert all(x.get('decoder_parity') for x in baseline)
print('BASELINE_VAL4_PASS',len(baseline),'points')
for method in CONTROL_METHODS.split(','):
    rs=[x['summary'] for x in baseline if x['method']==method]
    print('BASELINE',method,'F1',round(mean(x['mean_frame_f1'] for x in rs),6),
          'J',round(mean(x['objective'] for x in rs),6),
          'kbps',round(mean(x['actual_kbps'] for x in rs),3))

In [ ]:
def validate_point(results,target):
    assert len(results)==4,'A milestone must use all four locked Val videos'
    assert len({x['video'] for x in results})==4
    assert all(x['rate_kbps']==VAL_RATE and x['method']=='trained_d3qn' for x in results)
    assert all(x.get('decoder_parity') for x in results)
    f1=mean(x['summary']['mean_frame_f1'] for x in results)
    objective=mean(x['summary']['objective'] for x in results)
    kbps=mean(x['summary']['actual_kbps'] for x in results)
    hist=[sum(x['summary']['actions'][a] for x in results) for a in range(25)]
    assert sum(hist)==4*256
    roi_present=sum(x['summary'].get('roi_policy_diagnostics',{}).get('roi_present_frames',0)
                    for x in results)
    roi_absent=sum(x['summary'].get('roi_policy_diagnostics',{}).get('roi_absent_frames',0)
                   for x in results)
    assert roi_present+roi_absent==1024
    return {'steps':target,'results':results,
            'mean_f1':f1,'mean_objective':objective,'mean_actual_kbps':kbps,
            'action_histogram':hist,'dominant_action':int(max(range(25),key=lambda i:hist[i])),
            'dominant_fraction':max(hist)/sum(hist),
            'roi_present_frames':roi_present,'roi_absent_frames':roi_absent}

def current_best():
    if not monitor_history:return None
    return max(monitor_history,key=lambda x:x['mean_objective'])

def save_progress():
    savejson(TRAIN_HISTORY,train_history)
    savejson(MONITOR_HISTORY,monitor_history)
    winner=current_best()
    savejson(MANIFEST,{
        'schema':'d3qn-v3-mini16-val4-learning-check/1',
        'protocol':'native-masked-d3qn-qp/v3',
        'source_commit':SOURCE_COMMIT,'seed':SEED,
        'train_video_count':16,'validation_video_count':4,'test_used':False,
        'monitor_rate_kbps':VAL_RATE,
        'completed_steps':int(train_history[-1]['total_steps']) if train_history else 0,
        'finished_milestones':[int(x['steps']) for x in monitor_history],
        'best_steps':winner['steps'] if winner else None,
        'best_objective':winner['mean_objective'] if winner else None,
        'status':'complete' if len(monitor_history)==5 else 'partial'})
    
# Run in independent 1024-transition chunks to retain all 5 checkpoints.
for milestone in MILESTONES:
    if any(x['steps']==milestone for x in monitor_history):
        print('MONITOR_ALREADY_COMPLETE',milestone)
        continue
    completed_steps=int(train_history[-1]['total_steps']) if train_history else 0
    assert completed_steps<=milestone,'Unexpected training/monitor checkpoint mismatch'
    if completed_steps<milestone:
        run_dir=RUN/f'train_{completed_steps}_to_{milestone}'
        assert not run_dir.exists(),'Incomplete train run exists; inspect before retry'
        live(runner_args('train',run_dir,checkpoint=LATEST if completed_steps else None,
                         train_steps=milestone-completed_steps),REPO)
        progress=loadjson(run_dir/'result.json')
        newrows=loadjson(run_dir/'train_metrics.json')
        assert newrows and newrows[-1]['total_steps']>completed_steps
        train_history.extend(newrows)
        shutil.copyfile(run_dir/'checkpoint.pt',LATEST)
        save_progress()
        completed_steps=train_history[-1]['total_steps']
        print('TRAIN_PROGRESS_SAVED',completed_steps,'of',milestone)
        if progress['status']!='train_complete' or completed_steps!=milestone:
            print('TRAINING_SOFT_STOP: checkpoint saved; resume in another session')
            break
    val_dir=RUN/f'validation_{milestone}'
    assert not val_dir.exists(),'Incomplete Val4 stage exists; inspect before retry'
    live(runner_args('validate',val_dir,checkpoint=LATEST,
                     methods='trained_d3qn'),REPO)
    if loadjson(val_dir/'result.json')['status']!='validation_complete':
        print('VALIDATION_SOFT_STOP: checkpoint saved; resume later')
        break
    point=validate_point(loadjson(val_dir/'validation.json'),milestone)
    monitor_history.append(point)
    champion=current_best()
    if champion['steps']==milestone:
        shutil.copyfile(LATEST,BEST)
    save_progress()
    print('MONITOR_PASS',milestone,'F1',round(point['mean_f1'],6),
          'objective',round(point['mean_objective'],6),
          'kbps',round(point['mean_actual_kbps'],3),
          'dominant action',point['dominant_action'],
          'dominance',round(point['dominant_fraction'],3),
          'BEST',champion['steps'])
save_progress()
print('SESSION_END',loadjson(MANIFEST)['status'],'completed_steps',
      loadjson(MANIFEST)['completed_steps'])

In [ ]:
# Learning diagnostics on Val4. Note: validation selection is NOT Test30.
import numpy as np
import matplotlib.pyplot as plt

summary=[]
for m in monitor_history:
    summary.append({'step':m['steps'],'F1':m['mean_f1'],
                    'objective':m['mean_objective'],'kbps':m['mean_actual_kbps'],
                    'dominant_action':m['dominant_action'],
                    'dominant_fraction':m['dominant_fraction']})
print('MONITOR_SUMMARY',json.dumps(summary,indent=2))
if summary:
    x=[m['step'] for m in summary]
    qpa=[b['summary'] for b in baseline if b['method']=='qpa_anchor']
    fixed=[b['summary'] for b in baseline if b['method']=='global_minus2']
    roi19=[b['summary'] for b in baseline if b['method']=='roi_plus1_bg_plus2']
    b2=[b['summary'] for b in baseline if b['method']=='roi_minus2_bg_zero']
    fig,ax=plt.subplots(figsize=(8.5,4.8))
    ax.plot(x,[100*m['F1'] for m in summary],marker='o',label='V3 D3QN')
    for series,name in ((qpa,'Native QPA'),(fixed,'Global -2'),
                        (roi19,'Fixed (+1,+2)'),(b2,'Fixed (-2,0)')):
        ax.axhline(100*mean(r['mean_frame_f1'] for r in series),
                   linestyle='--',label=name)
    ax.set(xlabel='Training transitions',ylabel='Pseudo-detection frame F1 (%)',
           title='Mini Val4 @ 120 kbps — learning check')
    ax.grid(alpha=.25);ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(RUN/'val4_f1_curve.png',dpi=150)
    plt.show()
    plt.close(fig)
    fig,ax=plt.subplots(figsize=(8.5,4.8))
    ax.plot(x,[m['objective'] for m in summary],marker='o',label='V3 D3QN')
    for series,name in ((qpa,'Native QPA'),(fixed,'Global -2'),
                        (roi19,'Fixed (+1,+2)'),(b2,'Fixed (-2,0)')):
        ax.axhline(mean(r['objective'] for r in series),
                   linestyle='--',label=name)
    ax.set(xlabel='Training transitions',ylabel='Clip objective J',
           title='Objective: F1 - 0.15 × bitrate ratio - excess penalty')
    ax.grid(alpha=.25);ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(RUN/'val4_objective_curve.png',dpi=150)
    plt.show()
    plt.close(fig)
    fig,ax=plt.subplots(figsize=(8.5,4.8))
    ax.plot(x,[m['dominant_fraction']*100 for m in summary],marker='o')
    ax.set(xlabel='Training transitions',ylabel='Most frequent action (%)',
           ylim=(0,105),title='Greedy policy collapse diagnostic (Val4)')
    ax.grid(alpha=.25)
    fig.tight_layout()
    fig.savefig(RUN/'val4_policy_concentration.png',dpi=150)
    plt.show()
    plt.close(fig)

if train_history:
    print('TRAIN_EPISODES',len(train_history),'CURRENT_TRANSITIONS',
          train_history[-1]['total_steps'])
    ids={r['train_video'] for r in train_history}
    print('TRAIN_DISTINCT_VIDEOS',len(ids),'FIRST16_UNIQUE',
          len({r['train_video'] for r in train_history[:16]}))
    if len(train_history)>=16:
        assert len({r['train_video'] for r in train_history[:16]})==16
    print('LAST_5_EPISODE_UPDATES',[
        {'steps':r['total_steps'],'loss':r['mean_update']['loss'],
         'abs_td':r['mean_update']['mean_abs_td'],
         'q_gap':r['mean_update']['q_gap_valid'],
         'greedy_diversity':r['mean_update']['unique_greedy_actions_in_batch']}
        for r in train_history[-5:]])

In [ ]:
# Durable SAFE ZIP: both latest and BEST V3 checkpoint, all training/Val4 logs.
from IPython.display import FileLink,display
def export_safe():
    save_progress()
    paths=[TRAIN_HISTORY,MONITOR_HISTORY,BASELINE,MANIFEST,
           LATEST,BEST,RUN/'val4_f1_curve.png',
           RUN/'val4_objective_curve.png',RUN/'val4_policy_concentration.png',
           RUN/'native_preflight32'/'PREFLIGHT.json']
    tmp=SAFE.with_suffix('.zip.tmp')
    with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        for path in paths:
            if path.is_file():
                z.write(path,path.relative_to(RUN).as_posix())
        z.writestr('TOOLCHAIN.json',
                    json.dumps({k:toolchain[k] for k in ('source_commit',
                      'encoder_sha256','decoder_sha256') if k in toolchain},indent=2))
    tmp.replace(SAFE)
    print('D3QN_V3_SAFE_READY',SAFE,'MB',round(SAFE.stat().st_size/2**20,2))
    print('RUN_STATUS',loadjson(MANIFEST))
    return SAFE
output=export_safe()
os.chdir('/kaggle/working')
display(FileLink(output.name))
print('Download from Kaggle Output / Save Version if FileLink is unavailable.')

## Diễn giải sau khi chạy

**Chỉ chọn checkpoint tốt nhất trong năm mốc bằng Val4 objective**, không mặc định mốc 5.120 là tốt nhất. Để coi mô hình có dấu hiệu học, kiểm tra đồng thời:

1. F1 và objective trên Val4 **so với QPA và ba đối chứng cố định**.
2. So sánh **bitrate đo được**, không diễn giải cùng target bitrate như rate-matched.
3. Top-action share/ROI diagnostics: D3QN phải có lợi ích so với fixed (+1,+2), tránh chỉ tái tạo preset.
4. TD loss/Q-gap và hành vi greedy: không suy diễn chỉ từ loss giảm.
5. V3 phải được kiểm tra bằng **ablation n-step=1 vs3** nếu pilot tốt. Chưa Test30, chưa BD-rate, chưa có ground-truth mAP.

**Nếu phiên hết giờ:** tải SAFE ZIP, gắn lại làm Kaggle Input ở phiên mới, cấu hình `RESUME_SAFE_ZIP` trong cell đầu và chạy lại. Chỉ dùng checkpoint V3 cùng commit/config; runner sẽ kiểm tra identity.